
# Experiment Summary Aggregator (Notebook)
Dieses Notebook liest eine **Experiment_Summary**-CSV aus einem Basisordner ein  
und **hängt zusätzliche Daten** (Pfade & Metadaten) über *run_id* / *quant_mode* an, 
indem es die zugehörigen `ErrorMetrics_all_runs.csv` sowie JSON-Metrikdateien findet und verarbeitet.

## Was es macht
- Liest die Datei `Experiment_Summary_Serevr_multiconfig.csv` (oder kompatible Varianten) aus einem Basisordner.
- Vereinheitlicht Spalten (z. B. `level` vs. `profile`).
- Sucht pro `run_id` die Datei `Error_Metrics/ErrorMetrics_all_runs.csv` und joinet die passenden `json_path` / `predictions_file_path` je nach `quant_mode` an.
- Liest die JSON-Fehlerdateien und hängt **zusätzliche Metadaten** (Dataset, Zeitstempel, Trainings- & Inferenz-Config, Modellfilename etc.) an.
- Optional: berechnet aus den Metrik-Arrays sinnvolle Kennzahlen (z. B. Mittelwerte je Metrik und je gewünschtem `horizon`-Schritt).
- Speichert ein **angereichertes CSV** unter `Analysis/Experiment_Aggregated_Summary_enriched.csv` (wird automatisch angelegt).

> Hinweis: Dieses Notebook ist robust gegenüber beiden Summary-Varianten (Version 1/2).  
> Bitte die erste Zelle mit den **Parametern** anpassen und die Zellen nacheinander ausführen.


In [38]:

# === PARAMETER ===
# Basisordner der Output-Struktur (ohne abschließenden Backslash):
base_path = r"C:\DEV\RevPi_ML\ML_Edge_Device\Output"
base_path = r"C:\DEV\RevPi_ML\zwischenergebnisse_3"
base_path = r"C:\DEV\RevPi_ML\zwischenergebnisse"

# Dateiname der zu lesenden Summary:
summary_filename = "Experiment_Summary_Serevr_multiconfig.csv"
summary_filename = "Experiment_Summary_Serevr_multiconfig_RevPi.csv"
summary_filename = "Output\Error_Metrics\Experiment_Summary.csv"
# Ausgabeordner (wird bei Bedarf erstellt)
analysis_subdir = "Analysis"
enriched_csv_name = "Experiment_Aggregated_Summary_enriched.csv"

# === OPTIONEN ===
# Falls True: pro JSON-Metrik-Array auch den Wert zum jeweiligen 'horizon' extrahieren (Index horizon-1).
extract_horizon_specific = True

# Falls True: Mittelwerte je Metrik (über alle Horizonte in der JSON-Liste) mit anhängen.
compute_metric_means = True


In [73]:

import os
import json
import glob
import math
from pathlib import Path
import pandas as pd
from collections import defaultdict

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

def _normalize_quant_mode_from_variant_name(variant_name: str) -> str:
    # Bestimme quant_mode aus einer Variantenbezeichnung wie 'keras', 'model.keras',
    # 'tflite_model_quant_float16', 'model_quant_float16.tflite', 'model_quant_int8.tflite' usw.
    if not isinstance(variant_name, str):
        return ""
    v = variant_name.lower()
    if "int8" in v:
        return "quant-8"
    if "float16" in v or "fp16" in v:
        return "quant-16"
    if "keras" in v:
        return "no-quant"
    # Fallback: kein Treffer -> lieber leer lassen
    return ""

ALGO_TO_SUBDIR = {
    "cnn1d": "CNN1D",
    "lstm": "LSTM",
    "random_forest": "Random_Forest",
    "xgboost": "XGBoost",
    "light_xgboost": "Light_XGBoost",
    "lightgbm": "LightGBM",
}

def _safe_path(*parts) -> Path:
    return Path(*parts)

def _find_all_runs_csv(base: Path, run_id: str, algo: str = None) -> Path | None:
    """
    Suche die Datei 'Error_Metrics/ErrorMetrics_all_runs.csv' zu einem run_id.
    Versucht zuerst den Algorithmus-Ordner, danach globale Suche (langsamer).
    """
    # 1) gezielter Versuch über Algo-Unterordner
    if algo:
        subdir = ALGO_TO_SUBDIR.get(str(algo).lower(), None)
        if subdir:
            candidate = base / subdir / run_id / "Error_Metrics" / "ErrorMetrics_all_runs.csv"
            if candidate.exists():
                return candidate

    # 2) Fallback: globale Suche ab base
    pattern = str(base / "**" / run_id / "Error_Metrics" / "ErrorMetrics_all_runs.csv")
    matches = glob.glob(pattern, recursive=True)
    if matches:
        return Path(matches[0])

    return None

def _read_summary_csv(path: Path) -> pd.DataFrame:
    # sep=None + engine='python' versucht Delimiter zu erkennen (Komma/Semikolon etc.)
    df = pd.read_csv(path, sep=None, engine="python")
    # Vereinheitliche Spaltennamen
    cols = [c.strip() for c in df.columns]
    df.columns = cols
    # Version 2 hat 'profile' statt 'level'
    if "level" not in df.columns and "profile" in df.columns:
        df["level"] = df["profile"]
    # Lowercase algorithm
    if "algorithm" in df.columns:
        df["algorithm"] = df["algorithm"].astype(str).str.lower()
    # quant_mode in einheitliches Format
    if "quant_mode" in df.columns:
        df["quant_mode"] = df["quant_mode"].astype(str).str.lower()
    # Korrigiere Datentypen, wo sinnvoll
    for num_col in ["lags", "horizon"]:
        if num_col in df.columns:
            df[num_col] = pd.to_numeric(df[num_col], errors="coerce")
    return df

def _load_all_runs_df(all_runs_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(all_runs_csv)
    # Erwartete Spalten: run_id, model_name, dataset, time_stamp, model_variant, json_path, predictions_file_path
    # Erzeuge eine Spalte quant_mode_normalized aus model_variant
    if "model_variant" in df.columns:
        df["quant_mode_normalized"] = df["model_variant"].map(_normalize_quant_mode_from_variant_name)
    else:
        df["quant_mode_normalized"] = ""
    return df

def _read_json_safely(path: Path) -> dict:
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        return {"__error__": str(e), "__path__": str(path)}

def _flatten_training_inference_config(payload: dict) -> dict:
    # Ziehe aus training_config / inference_config nützliche Felder in flacher Struktur heraus.
    out = {}
    if not isinstance(payload, dict):
        return out

    # run info
    run = payload.get("run", {})
    for k in ["run_id", "model_name", "dataset", "time_stamp"]:
        if k in run:
            out[f"run.{k}"] = run[k]

    # training_config
    tr = payload.get("training_config", {})
    simple_keys = [
        "dataset","train_fraction","validation_fraction","rolling_window_size","lags",
        "batch_size","epochs","learning_rate","optimizer","loss","clipnorm",
        "cnn_dropout","cnn_activation","edge_device","enable_edge","train_time_s",
        "model_size_MB","training_time_s"
    ]
    for k in simple_keys:
        if k in tr:
            out[f"training_config.{k}"] = tr[k]

    # inference_config (nur ein paar Kernfelder)
    inf = payload.get("inference_config", {})
    for k in ["inference_steps","retraining_interval_steps","retraining_cycles",
              "inference_interval_sec","horizon","model_name","model_filename"]:
        if k in inf:
            out[f"inference_config.{k}"] = inf[k]

    # extra info
    extra = payload.get("extra_info", {})
    if isinstance(extra, dict):
        for k,v in extra.items():
            out[f"extra_info.{k}"] = v

    return out

def _extract_metrics(payload: dict, horizon: int | float | None, compute_means=True, extract_specific=True) -> dict:
    # Aus payload['metrics'] Metriken aggregieren.
    # - compute_means: Mittelwert je Metrik-Liste berechnen
    # - extract_specific: Wert an Index (horizon-1), falls vorhanden, ebenfalls anfügen
    out = {}
    metrics = payload.get("metrics", {})
    if not isinstance(metrics, dict):
        return out

    # Liste möglicher Feldnamen (je nach JSON variabel, daher dynamisch)
    for mname, mval in metrics.items():
        if isinstance(mval, list) and mval and all(isinstance(x, (int, float)) or (isinstance(x, float) and math.isnan(x)) for x in mval):
            series = pd.Series(mval, dtype="float64")

            if compute_means:
                out[f"metrics_mean.{mname}"] = float(series.mean(skipna=True))

            if extract_specific and horizon is not None and not pd.isna(horizon):
                idx = int(horizon) - 1
                if 0 <= idx < len(series):
                    out[f"metrics_h{int(horizon)}.{mname}"] = float(series.iloc[idx])
        else:
            # Einzelwerte (z. B. weighted_mae)
            if isinstance(mval, (int, float)) and not isinstance(mval, bool):
                out[f"metrics_value.{mname}"] = float(mval) if not pd.isna(mval) else mval

    return out


In [71]:

# === LADE SUMMARY ===
from pathlib import Path

base = Path(base_path)
summary_path = base / summary_filename
if not summary_path.exists():
    raise FileNotFoundError(f"Summary-Datei nicht gefunden: {summary_path}")

df_sum = _read_summary_csv(summary_path).copy()

# Einheitliche Mindestspalten prüfen
required_cols = ["algorithm","lags","horizon","model_variant","quant_mode","run_id"]
missing = [c for c in required_cols if c not in df_sum.columns]
if missing:
    raise ValueError(f"Fehlende Spalten in Summary: {missing}")

# === JOIN: json_path / predictions_file_path via ErrorMetrics_all_runs.csv ===
cache_all_runs = {}  # run_id -> DataFrame
json_paths = []
pred_paths = []

for idx, row in df_sum.iterrows():
    algo = str(row["algorithm"]).lower() if "algorithm" in row else None
    run_id = str(row["run_id"])
    qmode = str(row["quant_mode"]).lower()

    # Lade/cashe ErrorMetrics_all_runs.csv für diesen run_id
    if run_id not in cache_all_runs:
        all_runs_csv = _find_all_runs_csv(base, run_id, algo=algo)
        if all_runs_csv is None:
            cache_all_runs[run_id] = pd.DataFrame()
        else:
            cache_all_runs[run_id] = _load_all_runs_df(all_runs_csv)

    df_ar = cache_all_runs[run_id]
    json_path_val, pred_path_val = None, None
    if not df_ar.empty:
        # Match per quant_mode
        cand = df_ar[df_ar["quant_mode_normalized"] == qmode]
        if cand.empty:
            # Fallback: versuche exakter Variantenname (robustheit)
            mv = str(row["model_variant"])
            cand = df_ar[df_ar["model_variant"].astype(str).str.lower() == mv.lower()]
        if not cand.empty:
            # Nimm die erste passende Zeile
            json_path_val = cand.iloc[0].get("json_path", None)
            pred_path_val = cand.iloc[0].get("predictions_file_path", None)

    json_paths.append(json_path_val)
    pred_paths.append(pred_path_val)

df_sum["json_path"] = json_paths
df_sum["predictions_file_path"] = pred_paths

print("Anzahl gefundener JSON-Pfade:", df_sum["json_path"].notna().sum())
print("Anzahl gefundener Prediction-Pfade:", df_sum["predictions_file_path"].notna().sum())

# === Lade JSONs & hänge Metadaten + Metrik-Aggregate an ===
flat_rows = []
for idx, row in df_sum.iterrows():
    payload = {}
    jpath = row.get("json_path", None)
    if isinstance(jpath, str) and jpath:
        p = Path(jpath)
        payload = _read_json_safely(p)

    flat = _flatten_training_inference_config(payload)
    metrics_extra = _extract_metrics(payload, horizon=row.get("horizon", None),
                                     compute_means=compute_metric_means,
                                     extract_specific=extract_horizon_specific)
    # Füge die Basiszeile hinzu + Anhänge
    base_data = row.to_dict()
    base_data.update(flat)
    base_data.update(metrics_extra)
    flat_rows.append(base_data)

df_enriched = pd.DataFrame(flat_rows)

# === Ausgabe / Speichern ===
analysis_dir = base / analysis_subdir
analysis_dir.mkdir(parents=True, exist_ok=True)
out_path = analysis_dir / enriched_csv_name
df_enriched.to_csv(out_path, index=False)
print(f"Gespeichert: {out_path}")

# Zeige einen schnellen Überblick
df_enriched.head(10)


Anzahl gefundener JSON-Pfade: 288
Anzahl gefundener Prediction-Pfade: 288
Gespeichert: C:\DEV\RevPi_ML\zwischenergebnisse\Analysis\Experiment_Aggregated_Summary_enriched.csv


,algorithm,profile,lags,horizon,model_variant,quant_mode,avg_inference_time_ms,avg_total_time_ms,avg_cpu_percent,avg_ram_percent,model_size_mb,run_id,level,json_path,predictions_file_path
0,light_xgboost,edge,1,1,model.joblib,no-quant,3.237102,39.458267,25.683851,72.037931,0.1724,2025-08-28_143234_6731_train,edge,None,None
1,light_xgboost,edge,1,4,model.joblib,no-quant,12.194584,48.872540,29.103549,58.675862,0.6854,2025-08-28_143324_9778_train,edge,None,None
2,light_xgboost,edge,1,7,model.joblib,no-quant,19.439718,55.513597,31.384494,50.655172,1.2024,2025-08-28_143419_8734_train,edge,None,None
3,light_xgboost,edge,1,10,model.joblib,no-quant,26.563401,62.742188,32.041284,50.000000,1.7079,2025-08-28_143509_1282_train,edge,None,None
4,light_xgboost,edge,1,13,model.joblib,no-quant,38.910019,75.859416,31.985040,50.100000,2.2245,2025-08-28_143557_1221_train,edge,None,None
5,light_xgboost,edge,1,16,model.joblib,no-quant,42.692663,78.294118,33.647271,50.600000,2.7333,2025-08-28_143647_9433_train,edge,None,None
6,light_xgboost,edge,4,1,model.joblib,no-quant,2.985136,48.718997,26.825494,48.700000,0.1765,2025-08-28_143738_7472_train,edge,None,None
7,light_xgboost,edge,4,4,model.joblib,no-quant,13.048979,58.529426,27.719636,49.761538,0.6994,2025-08-28_143823_1225_train,edge,None,None
8,light_xgboost,edge,4,7,model.joblib,no-quant,21.734500,68.450335,29.819649,49.700000,1.2078,2025-08-28_143909_3679_train,edge,None,None
9,light_xgboost,edge,4,10,model.joblib,no-quant,29.195492,74.825340,30.910457,49.900000,1.7296,2025-08-28_143957_7559_train,edge,None,None


In [ ]:

# Falls Sie das Ergebnis als Tabelle im Notebook sehen möchten:
try:
    from caas_jupyter_tools import display_dataframe_to_user
    display_dataframe_to_user("Experiment_Aggregated_Summary_enriched", df_enriched)
except Exception as e:
    print("Hinweis:", e)


Hinweis: No module named 'caas_jupyter_tools'


In [84]:
# === SINGLE-KACHEL-METRIKEN (robust) =========================================
import os, glob, json, math, re
from pathlib import Path
import numpy as np
import pandas as pd

# --- Eingaben anpassen ---
BASE = r"C:\DEV\RevPi_ML\zwischenergebnisse"
BASE = r"C:\DEV\RevPi_ML\ML_Edge_Device\Output"


SUMMARY_FILE = "Output\Error_Metrics\Experiment_Summary.csv"
SUMMARY_FILE = "Experiment_Summary_Serevr_multiconfig.csv"  # dein Summary
SUMMARY_FILE = "Experiment_Summary_Serevr_multiconfig.csv"  # dein Summary



OUT_FILE = "Experiment_Enriched_SingleMetricKachel.csv"

summary_path = Path(BASE) / SUMMARY_FILE
df = pd.read_csv(summary_path)

# Horizon robust ermitteln
H_series = pd.to_numeric(df.get("horizon_num", df.get("horizon")), errors="coerce")

# Formatter
def _fmt(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return ""
        return f"{float(x):.6g}"
    except Exception:
        return str(x)

# Modellordner je Algorithmus
FOLDER_BY_ALGO = {"cnn1d": "CNN1D", "lstm": "LSTM", "random_forest": "Random_Forest"}

# --- robuste Variantenerkennung / Tokenisierung ------------------------------
def variant_tokens(model_variant: str) -> list[str]:
    mv = (model_variant or "").lower()
    toks = set()
    if not mv:
        return []
    # Grundtoken
    toks |= {mv, mv.replace(".", "_"), os.path.splitext(mv)[0], os.path.basename(mv)}
    # Spezielle Fälle
    if mv.endswith(".keras") or mv == "model.keras":
        toks |= {"keras", "model.keras", "model_keras"}
    if "int8" in mv:
        toks |= {"int8", "quant_int8", "model_quant_int8", "model_quant_int8.tflite", "tflite", "tflite_int8"}
    if "float16" in mv or "fp16" in mv:
        toks |= {"float16", "fp16", "quant_float16", "model_quant_float16", "model_quant_float16.tflite", "tflite", "tflite_float16"}
    if mv.endswith(".joblib") or "joblib" in mv or "sklearn" in mv:
        toks |= {"joblib", "sklearn"}
    return sorted(toks)

def safe_json_load(p: str):
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

# JSON-Pfad je Zeile bestimmen (robust)
def resolve_json_path(row) -> str | None:
    # 1) Falls Spalte vorhanden und Datei existiert → direkt verwenden
    if "json_path" in row.index:
        jp = row.get("json_path")
        if isinstance(jp, str) and jp and os.path.isfile(jp):
            return jp

    algo = str(row.get("algorithm","")).lower()
    folder = FOLDER_BY_ALGO.get(algo, algo.upper())
    run_id = str(row.get("run_id") or "")
    if not run_id:
        return None

    # Basisordner
    error_dir = Path(BASE) / folder / run_id / "Error_Metrics"
    if not error_dir.exists():
        # Fallback: suche beliebig tief unter BASE nach Run-Ordner/Error_Metrics
        pattern = str(Path(BASE) / "**" / run_id / "Error_Metrics")
        candidates = [Path(p) for p in glob.glob(pattern, recursive=True) if os.path.isdir(p)]
        if candidates:
            error_dir = candidates[0]
        else:
            return None

    # Token vorbereiten
    level = str(row.get("level","")).lower()
    ds = str(row.get("dataset","mqtt_data_filtered.csv"))
    ds_name = os.path.splitext(os.path.basename(ds))[0].lower()
    mv = str(row.get("model_variant",""))
    mv_tokens = variant_tokens(mv)

    # Erwarteter Name (best case)
    def expected_path():
        # versuche mehrere Suffixvarianten
        suffixes = mv_tokens or ["keras","model.keras","model_quant_int8.tflite","model_quant_float16.tflite","joblib"]
        for suf in suffixes:
            name = f"ErrorMetrics_{run_id}_{algo}_{level}_{ds_name}__{suf.replace('.','_')}.json"
            p = error_dir / name
            if p.is_file():
                return str(p)
        return None

    exp = expected_path()
    if exp:
        return exp

    # 2) Fallback: bestes Match im Ordner suchen
    cand = glob.glob(str(error_dir / "ErrorMetrics_*.json"))
    if not cand:
        return None

    # Scoring: Run-ID sehr hoch gewichten, dann algo/level/ds, dann variant tokens
    def score_for(path: str) -> tuple[int,int]:
        name = os.path.basename(path).lower()
        s = 0
        if run_id.lower() in name: s += 100
        for t in filter(None, [algo, level, ds_name]):
            if t in name: s += 10
        for t in mv_tokens:
            if t in name: s += 3
        return (s, -len(name))  # bei Gleichstand kürzerer Name bevorzugt

    cand.sort(key=lambda p: score_for(p), reverse=True)

    # Validierung: falls möglich, Run-ID im JSON prüfen
    for p in cand[:5]:
        payload = safe_json_load(p)
        rid = (((payload or {}).get("run") or {}).get("run_id") or "").lower()
        if run_id.lower() == rid:
            return p

    return cand[0] if cand else None

METRICS = ["mse","rmse","mae","r2","mape","smape","wape","msle","median_ae","mase","weighted_mae"]

def extract_metrics(json_path: str, H: int) -> dict:
    out = {}
    if not json_path or not os.path.isfile(json_path):
        return out
    payload = safe_json_load(json_path)
    if not payload:
        return out

    # Struktur: { "metrics": { ... } }  (so liegt deine Datei vor)
    metrics = (payload.get("metrics") or {})
    if not isinstance(metrics, dict):
        return out

    for k in METRICS:
        if k not in metrics:
            continue
        v = metrics[k]
        col = f"metrics_value.{k}"

        # Listen → auf H begrenzen, NaN/Inf filtern, als eine Zelle "(v1,v2,...)"
        if isinstance(v, (list, tuple)):
            vals = list(v)
            if isinstance(H, (int, np.integer)) and H > 0:
                vals = vals[:H]
            clean = [x for x in vals if not (isinstance(x, float) and (math.isnan(x) or math.isinf(x)))]
            out[col] = "(" + ",".join(_fmt(x) for x in clean) + ")" if clean else ""

        else:
            # Skalar → NaN/Inf zu leerem Feld
            if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
                out[col] = ""
            else:
                out[col] = _fmt(v)
    return out

rows = []
missing_json = 0
for i, row in df.iterrows():
    jp = resolve_json_path(row)
    if not jp:
        missing_json += 1
    H = H_series.iat[i] if i < len(H_series) else None
    H_int = int(H) if (pd.notna(H) and float(H).is_integer()) else 0
    rows.append(extract_metrics(jp, H_int))

metrics_df = pd.DataFrame(rows)

# Zusammenführen
df_out = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)

# Störende Altspalten entfernen, NEUE metrics_value.* behalten
drop_patterns = [r"^metrics_mean\.", r"^metrics_h\d+\.", r"^metrics_value\..*\.h\d+$"]
to_drop = []
for c in df_out.columns:
    for pat in drop_patterns:
        if re.match(pat, c):
            to_drop.append(c); break
if to_drop:
    df_out = df_out.drop(columns=sorted(set(to_drop)))

# Speichern
out_path = Path(BASE) / "Analysis" / OUT_FILE
out_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_path, index=False, encoding="utf-8")

added_cols = [c for c in df_out.columns if c.startswith("metrics_value.")]
print(f"OK: {len(added_cols)} metrics_value-Spalten hinzugefügt → {out_path}")
if missing_json:
    print(f"Warnung: {missing_json} Zeilen ohne (ableitbaren) JSON-Pfad – Metriken dort leer.")
df_out.head(3)


OK: 11 metrics_value-Spalten hinzugefügt → C:\DEV\RevPi_ML\zwischenergebnisse_3\Analysis\Experiment_Enriched_SingleMetricKachel.csv


,algorithm,level,lags,horizon,model_variant,quant_mode,avg_inference_time_ms,avg_total_time_ms,avg_cpu_percent,avg_ram_percent,model_size_mb,run_id,metrics_value.mse,metrics_value.rmse,metrics_value.mae,metrics_value.r2,metrics_value.mape,metrics_value.smape,metrics_value.wape,metrics_value.msle,metrics_value.median_ae,metrics_value.mase,metrics_value.weighted_mae
0,cnn1d,simple,20,1,model.keras,no-quant,183.099364,272.931780,25.162414,57.385246,0.3485,2025-08-29_111301_9032_train,3.05047,1.74656,1.74656,0,1.74656e+10,200,1.04794e+12,1.0208,1.74656,,1.74656
1,cnn1d,simple,20,1,model_quant_float16.tflite,quant-16,0.551254,85.597657,25.094267,45.188525,0.3485,2025-08-29_111301_9032_train,3.05265,1.74718,1.74718,0,1.74718e+10,200,1.04831e+12,1.02126,1.74718,,1.74718
2,cnn1d,simple,20,1,model_quant_int8.tflite,quant-8,0.559658,85.443419,25.329956,44.965574,0.3485,2025-08-29_111301_9032_train,3.1828,1.78404,1.78404,0,1.78404e+10,200,1.07042e+12,1.04838,1.78404,,1.78404


In [88]:
# === PREDICTIONS -> Serien + MAE/R² je Horizon (1..N) + Aggregationen (avg & pooled) ======
import os, glob, re, math
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------------------
# Eingaben
BASE = r"C:\DEV\RevPi_ML\ML_Edge_Device\Output"
SUMMARY_FILE = "Experiment_Summary_Server_multiconfig.csv"
OUT_FILE = "Experiment_Enriched_SingleMetricKachel.csv"

summary_path = Path(BASE) / SUMMARY_FILE
df = pd.read_csv(summary_path)

# Horizon robust laden
H_series = pd.to_numeric(df.get("horizon_num", df.get("horizon")), errors="coerce")

# -------------------------- Pfad-Resolver --------------------------------
FOLDER_BY_ALGO = {
    "cnn1d": "CNN1D", "lstm": "LSTM", "random_forest": "Random_Forest",
    "light_xgboost": "Light_XGBoost", "xgboost": "XGBoost"
}

def variant_tokens(model_variant: str) -> list[str]:
    mv = (model_variant or "").lower()
    toks = set()
    if not mv: return []
    toks |= {mv, mv.replace(".", "_"), os.path.splitext(mv)[0], os.path.basename(mv)}
    if mv.endswith(".keras") or mv == "model.keras":
        toks |= {"keras","model.keras","model_keras"}
    if "int8" in mv:
        toks |= {"int8","quant_int8","model_quant_int8","model_quant_int8.tflite",
                 "tflite","tflite_int8","tflite_model_quant_int8"}
    if "float16" in mv or "fp16" in mv:
        toks |= {"float16","fp16","quant_float16","model_quant_float16",
                 "model_quant_float16.tflite","tflite","tflite_float16",
                 "tflite_model_quant_float16"}
    if mv.endswith(".joblib") or "joblib" in mv or "sklearn" in mv:
        toks |= {"joblib","sklearn"}
    return sorted(toks)

def resolve_pred_csv_path(row) -> str | None:
    algo = str(row.get("algorithm","")).lower()
    folder = FOLDER_BY_ALGO.get(algo, algo.upper())
    run_id = str(row.get("run_id") or "")
    if not run_id:
        return None

    pred_dir = Path(BASE) / folder / run_id / "Prediction_Data"
    if not pred_dir.exists():
        pattern = str(Path(BASE) / "**" / run_id / "Prediction_Data")
        cands = [Path(p) for p in glob.glob(pattern, recursive=True) if os.path.isdir(p)]
        if cands:
            pred_dir = cands[0]
        else:
            return None

    ds = str(row.get("dataset", "mqtt_data_filtered.csv"))
    ds_name = os.path.splitext(os.path.basename(ds))[0].lower()
    mv_tokens = variant_tokens(str(row.get("model_variant","")))
    algo_label = FOLDER_BY_ALGO.get(algo, algo.upper())

    # Best-Guess-Dateiname
    suf_candidates = mv_tokens or [
        "keras","tflite_model_quant_int8","tflite_model_quant_float16","joblib"
    ]
    for suf in suf_candidates:
        name = f"StepPredictions_{run_id}_{algo_label}_{ds_name}__{suf}.csv"
        p = pred_dir / name
        if p.is_file():
            return str(p)

    # Fallback: scoren
    cand = glob.glob(str(pred_dir / f"StepPredictions_*{run_id}*{ds_name}*.csv")) or \
           glob.glob(str(pred_dir / "StepPredictions_*.csv"))
    if not cand:
        return None

    def score_for(path: str) -> tuple[int,int]:
        name = os.path.basename(path).lower()
        s = 0
        if run_id.lower() in name: s += 100
        if algo_label.lower() in name: s += 30
        if ds_name in name: s += 20
        for t in mv_tokens:
            if t in name: s += 5
        return (s, -len(name))
    cand.sort(key=lambda p: score_for(p), reverse=True)
    return cand[0]

# ---------------------------- Hilfsfunktionen -----------------------------
def _safe_arrays(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    return y_true[m], y_pred[m]

def mae(y, yhat):
    y, yhat = _safe_arrays(y, yhat)
    return float(np.mean(np.abs(yhat - y))) if y.size else float("nan")

def r2(y, yhat):
    y, yhat = _safe_arrays(y, yhat)
    if y.size == 0:
        return float("nan")
    ss_res = np.sum((yhat - y)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return float(1.0 - ss_res / ss_tot) if ss_tot > 0 else 0.0

def _fmt(x):
    try:
        if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
            return ""
        return f"{float(x):.6g}"
    except Exception:
        return str(x)

def _series_to_tuple_string(s: pd.Series) -> str:
    s = pd.to_numeric(s, errors="coerce")
    vals = [_fmt(v) for v in s if pd.notna(v)]
    return "(" + ",".join(vals) + ")" if vals else ""

# ------------------- Ausrichtung (Horizon) --------------------------------
# "t_plus_h":           pred_hh(t)  vs true_value(t+h)   => shift(-h)
# "t_plus_h_minus_1":   pred_hh(t)  vs true_value(t+h-1) => shift(-(h-1))
SHIFT_MODE = "t_plus_h"
def _true_for_h(series_true: pd.Series, h: int) -> pd.Series:
    return series_true.shift(-(h-1)) if SHIFT_MODE == "t_plus_h_minus_1" else series_true.shift(-h)

# ------------------- Kern: eine Zeile (ein Run) verarbeiten ----------------
def process_run_for_N(row, N: int):
    out = {
        # Ziel-H (N):
        "mae_from_preds": float("nan"),
        "r2_from_preds": float("nan"),
        # Aggregationen über h=1..N:
        "mae_all_avg_from_preds": float("nan"),
        "mae_all_pooled_from_preds": float("nan"),
        "r2_all_avg_from_preds": float("nan"),
        "r2_all_pooled_from_preds": float("nan"),
        # Serien:
        "true_value_series": "",
    }
    p = resolve_pred_csv_path(row)
    if not p:
        return out

    dfp = pd.read_csv(p)
    if "date" in dfp.columns:
        try: dfp = dfp.sort_values("date")
        except Exception: pass

    # Serien: true
    if "true_value" in dfp.columns:
        out["true_value_series"] = _series_to_tuple_string(dfp["true_value"])

    # Welche pred_hk gibt es?
    avail = sorted(int(m.group(1)) for c in dfp.columns if (m := re.match(r"pred_h(\d+)$", c)))
    if not avail:
        return out

    # Wir nutzen GENAU die Horizons 1..N, sofern vorhanden
    used_horizons = [h for h in range(1, int(N)+1) if f"pred_h{h}" in dfp.columns]
    if not used_horizons:
        return out

    y_true_all = pd.to_numeric(dfp.get("true_value"), errors="coerce")

    # Container für "pooled" (über alle Horizons gestapelt)
    pooled_true = []
    pooled_pred = []

    # Serien & MAE/R² je Horizon (nur h in 1..N)
    mae_vals = []
    r2_vals = []

    for h in used_horizons:
        col = f"pred_h{h}"
        # Serien speichern
        out[f"pred_h{h}_series"] = _series_to_tuple_string(dfp[col])

        # MAE/R² für h
        y_pred_h = pd.to_numeric(dfp[col], errors="coerce")
        y_true_h = pd.to_numeric(_true_for_h(y_true_all, h), errors="coerce")
        mask = y_pred_h.notna() & y_true_h.notna()
        y = y_true_h[mask].to_numpy()
        yhat = y_pred_h[mask].to_numpy()

        mae_h = mae(y, yhat)
        r2_h = r2(y, yhat)

        out[f"mae_h{h}_from_preds"] = mae_h
        out[f"r2_h{h}_from_preds"] = r2_h

        if np.isfinite(mae_h): mae_vals.append(mae_h)
        if np.isfinite(r2_h):  r2_vals.append(r2_h)

        # für pooled
        if y.size:
            pooled_true.append(y)
            pooled_pred.append(yhat)

    # Durchschnitt (Alle Horizons gleich gewichtet)
    if mae_vals:
        out["mae_all_avg_from_preds"] = float(np.mean(mae_vals))
    if r2_vals:
        out["r2_all_avg_from_preds"] = float(np.mean(r2_vals))

    # Gepoolt (alle (y,ŷ) über h=1..N stapeln)
    if pooled_true and pooled_pred:
        y_pool  = np.concatenate(pooled_true, axis=0)
        yhat_pool = np.concatenate(pooled_pred, axis=0)
        out["mae_all_pooled_from_preds"] = mae(y_pool, yhat_pool)
        out["r2_all_pooled_from_preds"]  = r2(y_pool, yhat_pool)

    # MAE & R² für Ziel-Horizon N (nur, wenn pred_hN existiert)
    target_col = f"pred_h{int(N)}"
    if target_col in dfp.columns:
        y_pred_N = pd.to_numeric(dfp[target_col], errors="coerce")
        y_true_N = pd.to_numeric(_true_for_h(y_true_all, int(N)), errors="coerce")
        maskN = y_pred_N.notna() & y_true_N.notna()
        yN = y_true_N[maskN].to_numpy()
        yhatN = y_pred_N[maskN].to_numpy()
        out["mae_from_preds"] = mae(yN, yhatN)
        out["r2_from_preds"]  = r2(yN, yhatN)

    return out

# --------------------- Schleife über Summary & Anreicherung ----------------
rows = []
missing_preds = 0
for i, row in df.iterrows():
    H = H_series.iat[i] if i < len(H_series) else None
    N = int(H) if (pd.notna(H) and float(H).is_integer()) else 1
    res = process_run_for_N(row, N)
    if np.isnan(res.get("mae_from_preds", np.nan)) and res.get("true_value_series","") == "":
        missing_preds += 1
    rows.append(res)

metrics_df = pd.DataFrame(rows)

# Zusammenführen
df_out = pd.concat([df.reset_index(drop=True), metrics_df], axis=1)

# Optional: alte metrics_value.*-Spalten entfernen
drop_patterns = [r"^metrics_value\.", r"^metrics_mean\.", r"^metrics_h\d+\."]
to_drop = [c for c in df_out.columns if any(re.match(pat, c) for pat in drop_patterns)]
if to_drop:
    df_out = df_out.drop(columns=sorted(set(to_drop)))

# Speichern
out_path = Path(BASE) / "Analysis" / OUT_FILE
out_path.parent.mkdir(parents=True, exist_ok=True)
df_out.to_csv(out_path, index=False, encoding="utf-8")

print(f"OK: Serien (true + pred_h1..hN), MAE/R² je h<=N, Aggregationen (avg/pooled) + MAE/R²@N → {out_path}")
if missing_preds:
    print(f"Warnung: {missing_preds} Zeilen ohne auffindbare Predictions-CSV.")
df_out.head(3)


OK: Serien (true + pred_h1..hN), MAE/R² je h<=N, Aggregationen (avg/pooled) + MAE/R²@N → C:\DEV\RevPi_ML\ML_Edge_Device\Output\Analysis\Experiment_Enriched_SingleMetricKachel.csv
Warnung: 24 Zeilen ohne auffindbare Predictions-CSV.


,algorithm,level,lags,horizon,model_variant,quant_mode,avg_inference_time_ms,avg_total_time_ms,avg_cpu_percent,avg_ram_percent,model_size_mb,run_id,mae_from_preds,r2_from_preds,mae_all_avg_from_preds,mae_all_pooled_from_preds,r2_all_avg_from_preds,r2_all_pooled_from_preds,true_value_series,pred_h1_series,mae_h1_from_preds,r2_h1_from_preds,pred_h2_series,mae_h2_from_preds,r2_h2_from_preds,pred_h3_series,mae_h3_from_preds,r2_h3_from_preds,pred_h4_series,mae_h4_from_preds,r2_h4_from_preds,pred_h5_series,mae_h5_from_preds,r2_h5_from_preds,pred_h6_series,mae_h6_from_preds,r2_h6_from_preds,pred_h7_series,mae_h7_from_preds,r2_h7_from_preds,pred_h8_series,mae_h8_from_preds,r2_h8_from_preds,pred_h9_series,mae_h9_from_preds,r2_h9_from_preds,pred_h10_series,mae_h10_from_preds,r2_h10_from_preds,pred_h11_series,mae_h11_from_preds,r2_h11_from_preds,pred_h12_series,mae_h12_from_preds,r2_h12_from_preds,pred_h13_series,mae_h13_from_preds,r2_h13_from_preds,pred_h14_series,mae_h14_from_preds,r2_h14_from_preds,pred_h15_series,mae_h15_from_preds,r2_h15_from_preds
0,lstm,simple,20,1,model.keras,no-quant,60.415092,85.427217,9.022172,33.386768,0.3571,2025-08-30_180545_6241_train,0.936896,0.712973,0.936896,0.936896,0.712973,0.712973,"(3.51,3.33,1.49,0.85,4.16,2.98,0.59,0.64,2.83,...","(2.12508,1.41728,1.2051,5.05353,2.70526,1.0073...",0.936896,0.712973,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,lstm,simple,20,1,model_quant_float16.tflite,quant-16,0.887552,29.388251,8.190064,33.457050,0.3571,2025-08-30_180545_6241_train,0.936892,0.712975,0.936892,0.936892,0.712975,0.712975,"(3.51,3.33,1.49,0.85,4.16,2.98,0.59,0.64,2.83,...","(2.12541,1.41727,1.20486,5.05401,2.70542,1.007...",0.936892,0.712975,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,lstm,simple,20,1,model_quant_int8.tflite,quant-8,0.954366,27.455755,8.004714,33.428633,0.3571,2025-08-30_180545_6241_train,0.936671,0.712724,0.936671,0.936671,0.712724,0.712724,"(3.51,3.33,1.49,0.85,4.16,2.98,0.59,0.64,2.83,...","(2.11998,1.4213,1.2049,5.03446,2.71265,1.0222,...",0.936671,0.712724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
# === Scientific Plotly 3D · Overlay (no-quant vs. fp16) mit einheitlichem Kamerawinkel ===
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---------- Einstellungen (global konsistent) ----------
CAMERA_SCI = dict(eye=dict(x=1.55, y=1.55, z=1.05), center=dict(x=0, y=0, z=0), up=dict(x=0, y=0, z=1))
FONT_SCI   = dict(family="Arial, Helvetica, sans-serif", size=14)
GRIDCOLOR  = "rgba(0,0,0,0.12)"

# Farben (uniforme Surfaces – „Schicht über Schicht“)
COLOR_NOQ  = "#222222"   # dunkelgrau
COLOR_FP16 = "#1f77b4"   # blau

# ---------- Helper ----------
def _label_quant(q: str) -> str:
    s = str(q).lower()
    if "no-quant" in s or "no_quant" in s or s in {"none","no"}:
        return "no-quant"
    if re.search(r"(float\s*16|fp\s*16|quant.*16)", s):
        return "quant-fp16"
    return q

def _clean_df(df: pd.DataFrame) -> pd.DataFrame:
    need = {"algorithm","lags","horizon","avg_inference_time_ms"}
    miss = need - set(df.columns)
    if miss: raise ValueError(f"Fehlende Spalten: {sorted(miss)}")
    d = df.copy()
    d["lags"] = pd.to_numeric(d["lags"], errors="coerce")
    d["horizon"] = pd.to_numeric(d["horizon"], errors="coerce")
    d["avg_inference_time_ms"] = pd.to_numeric(d["avg_inference_time_ms"], errors="coerce")
    d["quant_mode"] = d.get("quant_mode", "no-quant")
    d["quant_mode"] = d["quant_mode"].map(_label_quant)
    return d.dropna(subset=["lags","horizon","avg_inference_time_ms"])

def _aggregate(df: pd.DataFrame) -> pd.DataFrame:
    return (df.groupby(["algorithm","quant_mode","lags","horizon"], as_index=False)
              ["avg_inference_time_ms"].mean())

def _to_grid(df_sub: pd.DataFrame, z_col: str = "avg_inference_time_ms"):
    piv = df_sub.pivot_table(index="lags", columns="horizon", values=z_col, aggfunc="mean")
    X = np.array(sorted(piv.columns))   # horizons
    Y = np.array(sorted(piv.index))     # lags
    Z = piv.reindex(index=Y, columns=X).values.astype(float)
    return X, Y, Z

def _apply_scientific_layout(fig: go.Figure, title: str, log_z: bool):
    scene = dict(
        xaxis=dict(title="Horizon",  showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                   ticks="outside", mirror=True, showline=True, linewidth=1, linecolor="black"),
        yaxis=dict(title="Lags",     showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                   ticks="outside", mirror=True, showline=True, linewidth=1, linecolor="black"),
        zaxis=dict(title="Inference [ms]", type="log" if log_z else "linear",
                   showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                   ticks="outside", mirror=True, showline=True, linewidth=1, linecolor="black",
                   exponentformat="e", tickformat=".2e" if log_z else None),
        bgcolor="white",
        camera=CAMERA_SCI,
    )
    fig.update_layout(
        title=title, template="simple_white", font=FONT_SCI,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=10, r=10, t=60, b=10), height=650
    )
    fig.update_scenes(**scene)

# ---------- Kernfunktion: Overlay-Surfaces im selben Graph ----------
def plot_quant_overlay_surface(df: pd.DataFrame,
                               algorithm: str,
                               pair=("no-quant","quant-fp16"),
                               log_z: bool = True) -> go.Figure:
    d = _clean_df(df)
    sub = d[(d["algorithm"] == algorithm) & (d["quant_mode"].isin(pair))].copy()
    if sub.empty:
        raise ValueError(f"Keine Daten für '{algorithm}' mit Quant-Paar {pair}.")

    g = _aggregate(sub)

    # Grids bauen (Surfaces erlauben „Schicht“-Darstellung)
    traces = []
    for q in pair:
        gq = g[g["quant_mode"] == q]
        if gq.empty:
            continue
        X, Y, Z = _to_grid(gq, "avg_inference_time_ms")

        # uniforme Farbe via surfacecolor + 2-Punkt-Colorscale
        color = COLOR_NOQ if q == "no-quant" else COLOR_FP16
        traces.append(go.Surface(
            x=X, y=Y, z=Z,
            name=q, showscale=False,
            opacity=0.88 if q == "no-quant" else 0.65,   # „Schicht über Schicht“
            surfacecolor=np.zeros_like(Z),
            colorscale=[[0, color], [1, color]],
            contours=dict(
                z=dict(show=True, usecolormap=False, highlightcolor="black", project_z=True)
            ),
            lighting=dict(ambient=0.6, diffuse=0.7, specular=0.1, roughness=0.5)
        ))

    # Fallback (falls ein Grid stark lückenhaft ist): Punkte zusätzlich einblenden
    fig = go.Figure(data=traces)
    for q in pair:
        gq = g[g["quant_mode"] == q]
        fig.add_trace(go.Scatter3d(
            x=gq["horizon"], y=gq["lags"], z=gq["avg_inference_time_ms"],
            mode="markers", name=f"{q} · Punkte",
            marker=dict(size=3, color="#000000" if q=="no-quant" else COLOR_FP16, opacity=0.55),
            showlegend=False
        ))

    _apply_scientific_layout(fig, f"{algorithm} · Overlay: {pair[0]} vs. {pair[1]}", log_z=log_z)
    return fig

# ---------- Schleife: für alle Algorithmen den Overlay-Plot erzeugen ----------
algos = sorted(df_out["algorithm"].unique())
for algo in algos:
    fig = plot_quant_overlay_surface(df_out, algorithm=algo,
                                     pair=("no-quant","quant-fp16"),
                                     log_z=True)  # log_z=False falls linear gewünscht
    fig.show()

# Optional: denselben Kamerawinkel später erneut erzwingen (z.B. nach Interaktion)
# for algo in algos:
#     fig = plot_quant_overlay_surface(df_out, algo)
#     fig.update_layout(scene_camera=CAMERA_SCI)
#     fig.show()


In [53]:
# === Plotly 3D · Overlay je Modell (no-quant + Quant) mit einheitlichem Kamerawinkel ===
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------- Wissenschaftliches Styling ----------
CAMERA_SCI = dict(eye=dict(x=1.55, y=1.55, z=1.05), center=dict(x=0, y=0, z=0), up=dict(x=0, y=0, z=1))
FONT_SCI   = dict(family="Arial, Helvetica, sans-serif", size=14)
GRIDCOLOR  = "rgba(0,0,0,0.12)"

COLORS = {
    "no-quant": "#222222",
    "quant-fp16": "#1f77b4",
    "quant-int8": "#d62728",
    # fallback:
    "other": "#2ca02c",
}

def _label_quant(q: str) -> str:
    s = str(q).lower()
    if "no" in s and "quant" in s:              return "no-quant"
    if re.search(r"(float\s*16|fp\s*16|quant.*16)", s):  return "quant-fp16"
    if re.search(r"(int\s*8|quant[-_ ]?8)", s):          return "quant-int8"
    return s

def _clean_df(df: pd.DataFrame) -> pd.DataFrame:
    need = {"algorithm","lags","horizon","avg_inference_time_ms"}
    miss = need - set(df.columns)
    if miss: raise ValueError(f"Fehlende Spalten: {sorted(miss)}")
    d = df.copy()
    d["lags"] = pd.to_numeric(d["lags"], errors="coerce")
    d["horizon"] = pd.to_numeric(d["horizon"], errors="coerce")
    d["avg_inference_time_ms"] = pd.to_numeric(d["avg_inference_time_ms"], errors="coerce")
    d["quant_mode"] = d.get("quant_mode", "no-quant")
    d["quant_mode"] = d["quant_mode"].map(_label_quant)
    return d.dropna(subset=["lags","horizon","avg_inference_time_ms"])

def _agg(df: pd.DataFrame) -> pd.DataFrame:
    return (df.groupby(["algorithm","quant_mode","lags","horizon"], as_index=False)
              ["avg_inference_time_ms"].mean())

def _to_grid(df_sub: pd.DataFrame, zcol="avg_inference_time_ms"):
    piv = df_sub.pivot_table(index="lags", columns="horizon", values=zcol, aggfunc="mean")
    X = np.array(sorted(piv.columns))   # horizons
    Y = np.array(sorted(piv.index))     # lags
    Z = piv.reindex(index=Y, columns=X).values.astype(float)
    return X, Y, Z

# ---------- z-Skalierung: 'log' | 'asinh' | 'cut' (Pseudo-broken axis) ----------
def transform_z(Z, mode="log", cut_value=None, compress=0.25):
    """
    - log: keine Transformation der Daten; Achse in Plotly auf log setzen
    - asinh: weiche Kompression: z' = asinh(z / s), s ~ Median -> linear für klein, log-ähnlich für groß
    - cut:   Pseudo-'broken axis': bis cut_value linear, danach kleinere Steigung (komprimiert)
    """
    if mode == "log":
        return Z, None  # Achse auf log, Daten unverändert

    if mode == "asinh":
        s = np.nanmedian(Z[Z>0]) or 1.0
        Zt = np.arcsinh(Z / s)
        # Ticks: zurückrechnen
        ticks = np.linspace(np.nanmin(Zt), np.nanmax(Zt), 6)
        tickvals = ticks
        ticktext = [f"{(np.sinh(t)*s):.2g}" for t in ticks]
        return Zt, dict(type="linear", tickmode="array", tickvals=tickvals, ticktext=ticktext)

    if mode == "cut":
        if cut_value is None:
            # default: 95%-Quantil der Werte als Cut
            finite = Z[np.isfinite(Z)]
            cut_value = float(np.quantile(finite, 0.95)) if finite.size else 1.0
        Zt = Z.copy()
        above = Z > cut_value
        Zt[above] = cut_value + (Z[above] - cut_value) * compress
        # Ticks: ein paar unterhalb + oberhalb
        low_ticks  = np.linspace(np.nanmin(Z), cut_value, 4)
        high_ticks = np.linspace(cut_value, np.nanmax(Z), 3)[1:]  # ohne Doppelung
        def _map(v):
            return cut_value + (v - cut_value) * compress if v > cut_value else v
        tickvals = [_map(v) for v in list(low_ticks) + list(high_ticks)]
        ticktext = [f"{v:.2g}" for v in list(low_ticks) + list(high_ticks)]
        return Zt, dict(type="linear", tickmode="array", tickvals=tickvals, ticktext=ticktext)

    raise ValueError("mode muss 'log', 'asinh' oder 'cut' sein")

def _apply_layout(fig: go.Figure, title: str, zaxis_style=None, log_axis=False, height=650):
    scene = dict(
        xaxis=dict(title="Horizon",  showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                   ticks="outside", mirror=True, showline=True, linewidth=1, linecolor="black"),
        yaxis=dict(title="Lags",     showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                   ticks="outside", mirror=True, showline=True, linewidth=1, linecolor="black"),
        zaxis=dict(title="Inference [ms]",
                   type="log" if log_axis else "linear",
                   showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                   ticks="outside", mirror=True, showline=True, linewidth=1, linecolor="black",
                   exponentformat="e"),
        bgcolor="white",
        camera=CAMERA_SCI,
    )
    if zaxis_style:
        scene["zaxis"].update(zaxis_style)

    fig.update_layout(
        title=title, template="simple_white", font=FONT_SCI,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=10, r=10, t=60, b=10), height=height
    )
    fig.update_scenes(**scene)

def plot_overlay_for_algorithm(df: pd.DataFrame,
                               algorithm: str,
                               include_quants=("no-quant","quant-fp16","quant-int8"),
                               z_mode="log",           # 'log' | 'asinh' | 'cut'
                               cut_value=None,         # nur für z_mode='cut'
                               compress=0.25):
    d = _clean_df(df)
    d = d[(d["algorithm"] == algorithm) & (d["quant_mode"].isin(include_quants))]
    if d.empty:
        raise ValueError(f"Keine Daten für '{algorithm}' mit {include_quants}.")

    g = _agg(d)
    fig = go.Figure()

    # Surfaces: Schicht über Schicht
    # Reihenfolge: zuerst no-quant (unten/opaque), dann Quant (oben/transparenter)
    order = [q for q in ("no-quant","quant-fp16","quant-int8") if q in g["quant_mode"].unique()]
    for q in order:
        sub = g[g["quant_mode"] == q]
        X, Y, Z = _to_grid(sub, "avg_inference_time_ms")

        # z-Transformation (falls asinh/cut)
        Zt, zstyle = transform_z(Z, mode=z_mode, cut_value=cut_value, compress=compress)

        color = COLORS.get(q, COLORS["other"])
        fig.add_trace(go.Surface(
            x=X, y=Y, z=Zt,
            name=q, showscale=False,
            opacity=0.88 if q == "no-quant" else 0.65,
            surfacecolor=np.zeros_like(Zt),
            colorscale=[[0, color], [1, color]],
            contours=dict(z=dict(show=True, usecolormap=False, highlightcolor="black", project_z=True)),
            lighting=dict(ambient=0.6, diffuse=0.7, specular=0.1, roughness=0.5),
        ))
        # Punkte als Referenz (falls Grid unvollständig)
        fig.add_trace(go.Scatter3d(
            x=sub["horizon"], y=sub["lags"], z=Zt.ravel(order="C")[:len(sub)],  # robust fallback
            mode="markers", name=f"{q} · Punkte", showlegend=False,
            marker=dict(size=2, color=color, opacity=0.5),
        ))

    # Layout
    _apply_layout(
        fig,
        title=f"{algorithm} · Overlay: no-quant + Quant",
        zaxis_style=(zstyle if z_mode in {"asinh","cut"} else None),
        log_axis=(z_mode == "log"),
    )
    # Achsentitel bei alternativen Modi anpassen
    if z_mode == "asinh":
        fig.update_scenes(zaxis_title="asinh(Inference / s)")
    if z_mode == "cut":
        fig.add_annotation(text="z-Achse komprimiert (Pseudo-Cut)",
                           xref="paper", yref="paper", x=0, y=1.07, showarrow=False,
                           font=dict(size=12, color="#444"))

    return fig

# ---------- Grid: alle Modelle, jeweils 2 nebeneinander ----------
def plot_overlay_grid_all_models(df: pd.DataFrame,
                                 include_quants=("no-quant","quant-fp16","quant-int8"),
                                 z_mode="log", cut_value=None, compress=0.25, cols=2):
    d = _clean_df(df)
    algos = sorted(d["algorithm"].unique())
    rows = int(np.ceil(len(algos)/cols))
    fig = make_subplots(
        rows=rows, cols=cols,
        specs=[[{"type":"scene"} for _ in range(cols)] for _ in range(rows)],
        subplot_titles=algos
    )

    for i, algo in enumerate(algos):
        r, c = divmod(i, cols)
        r, c = r+1, c+1
        sub = d[(d["algorithm"] == algo) & (d["quant_mode"].isin(include_quants))]
        if sub.empty:
            continue
        g = _agg(sub)

        order = [q for q in ("no-quant","quant-fp16","quant-int8") if q in g["quant_mode"].unique()]
        zstyle_scene = None
        for q in order:
            gq = g[g["quant_mode"] == q]
            X, Y, Z = _to_grid(gq)
            Zt, zstyle_scene = transform_z(Z, mode=z_mode, cut_value=cut_value, compress=compress)
            color = COLORS.get(q, COLORS["other"])
            fig.add_trace(go.Surface(
                x=X, y=Y, z=Zt, name=f"{algo} · {q}", showscale=False,
                opacity=0.88 if q == "no-quant" else 0.65,
                surfacecolor=np.zeros_like(Zt),
                colorscale=[[0, color], [1, color]],
                contours=dict(z=dict(show=True, usecolormap=False, highlightcolor="black", project_z=True)),
                lighting=dict(ambient=0.6, diffuse=0.7, specular=0.1, roughness=0.5),
            ), row=r, col=c)

        # Szene-Layout je Subplot (gleicher Kamerawinkel überall)
        scene_kw = dict(
            xaxis_title="Horizon", yaxis_title="Lags",
            zaxis_title="Inference [ms]",
            bgcolor="white",
            camera=CAMERA_SCI,
            zaxis=dict(
                type="log" if z_mode == "log" else "linear",
                showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                ticks="outside", mirror=True, showline=True, linewidth=1, linecolor="black",
                exponentformat="e",
            )
        )
        if z_mode in {"asinh","cut"} and zstyle_scene:
            scene_kw["zaxis"].update(zstyle_scene)
            if z_mode == "asinh":
                scene_kw["zaxis"]["title"] = "asinh(Inference / s)"
        fig.update_scenes(**scene_kw, row=r, col=c)

    fig.update_layout(
        title="Modelle (jeweils Overlay: no-quant + Quant) – einheitlicher Kamerawinkel",
        template="simple_white", font=FONT_SCI,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(l=10, r=10, t=60, b=10), height=max(650, rows*520)
    )
    if z_mode == "cut":
        fig.add_annotation(text="z-Achse komprimiert (Pseudo-Cut)",
                           xref="paper", yref="paper", x=0, y=1.07, showarrow=False,
                           font=dict(size=12, color="#444"))
    return fig

# ---------- Beispiel-Nutzung ----------
# 1) Je Modell ein Overlay-Plot (no-quant + alle vorhandenen Quant-Varianten), log-z:
for algo in sorted(df_out["algorithm"].unique()):
    fig = plot_overlay_for_algorithm(df_out, algorithm=algo,
                                     include_quants=("no-quant","quant-fp16","quant-int8"),
                                     z_mode="log")  # oder 'asinh' / 'cut'
    fig.show()

# 2) Alle Modelle in einem Raster (2 nebeneinander), hier mit Pseudo-Cut:
fig_grid = plot_overlay_grid_all_models(
    df_out,
    include_quants=("no-quant","quant-fp16","quant-int8"),
    z_mode="cut",       # 'log' für reine Log-Achse, 'asinh' für weiche Kompression
    cut_value=None,     # None = automatisch (95%-Quantil); sonst z.B. 50 (ms)
    compress=0.25,      # Kompressionsfaktor >0 & <1 (kleiner = stärker komprimiert)
    cols=2
)
fig_grid.show()


In [58]:
# === Plotly · y = Inference [ms], x = Lags@H=10 und Horizon@L=10 (gemeinsam) ===
import re
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ---- Style ----
FONT_SCI  = dict(family="Arial, Helvetica, sans-serif", size=14)
GRIDCOLOR = "rgba(0,0,0,0.12)"
COLOR_LAGS, COLOR_HOR = "#1f77b4", "#d62728"   # blau / rot
LINE_DASH = {"no-quant":"solid","quant-fp16":"dash","quant-int8":"dot"}
MARKERS   = {"no-quant":"circle","quant-fp16":"triangle-up","quant-int8":"square"}

# Fixpunkte
H_FIX = 10  # für Lags-Spur (Horizon fest)
L_FIX = 10  # für Horizon-Spur (Lags fest)

# ---- Helpers ----
def _label_quant(q: str) -> str:
    s = str(q).lower()
    if "no" in s and "quant" in s:                      return "no-quant"
    if re.search(r"(float\s*16|fp\s*16|quant.*16)", s): return "quant-fp16"
    if re.search(r"(int\s*8|quant[-_ ]?8)", s):         return "quant-int8"
    return s

def _clean_df(df: pd.DataFrame) -> pd.DataFrame:
    need = {"algorithm","lags","horizon","avg_inference_time_ms"}
    miss = need - set(df.columns)
    if miss: raise ValueError(f"Fehlende Spalten: {sorted(miss)}")
    d = df.copy()
    d["lags"] = pd.to_numeric(d["lags"], errors="coerce")
    d["horizon"] = pd.to_numeric(d["horizon"], errors="coerce")
    d["avg_inference_time_ms"] = pd.to_numeric(d["avg_inference_time_ms"], errors="coerce")
    d["quant_mode"] = d.get("quant_mode","no-quant").map(_label_quant)
    return d.dropna(subset=["lags","horizon","avg_inference_time_ms"])

def _agg(df: pd.DataFrame) -> pd.DataFrame:
    return (df.groupby(["algorithm","quant_mode","lags","horizon"], as_index=False)
              ["avg_inference_time_ms"].mean())

def _nearest_available(values, target):
    vals = np.array(sorted(set(values)))
    if vals.size == 0: return None
    if target in vals: return target
    return float(vals[np.argmin(np.abs(vals - target))])

def _apply_layout(fig: go.Figure, title: str, note: str = ""):
    fig.update_layout(
        title=title, template="simple_white", font=FONT_SCI,
        margin=dict(l=10, r=10, t=60, b=10), height=520,
        legend=dict(orientation="h", y=1.08, x=1, xanchor="right", yanchor="bottom")
    )
    # x: numerische Achse für *beide* Variablen (Lags & Horizon)
    fig.update_xaxes(title="x = Lags (blau) & Horizon (rot)",
                     showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                     ticks="outside", showline=True, linewidth=1, linecolor="black")
    # y: Inference (linear)
    fig.update_yaxes(title="Inference [ms]", type="linear",
                     showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
                     ticks="outside", showline=True, linewidth=1, linecolor="black")
    if note:
        fig.add_annotation(text=note, xref="paper", yref="paper", x=0, y=1.13,
                           showarrow=False, font=dict(size=12, color="#444"))
    fig.add_annotation(text="Lags = blau (H=10) · Horizon = rot (L=10)",
                       xref="paper", yref="paper", x=0, y=1.08,
                       showarrow=False, font=dict(size=12, color="#444"))

def _add_traces_xy(fig: go.Figure, dfq: pd.DataFrame, label: str,
                   h_fix: float, l_fix: float):
    dash   = LINE_DASH.get(label, "solid")
    marker = MARKERS.get(label, "circle")

    # --- Spur 1: x = Lags (bei Horizon = h_fix), y = Inference ---
    d_l = dfq[dfq["horizon"] == h_fix].copy()
    if not d_l.empty:
        d_l = d_l.sort_values("lags")
        fig.add_trace(go.Scatter(
            x=d_l["lags"], y=d_l["avg_inference_time_ms"],
            mode="lines+markers",
            name=f"{label} · Lags@H{int(h_fix)}",
            legendgroup=f"{label}-lags", showlegend=True,
            line=dict(color=COLOR_LAGS, width=2, dash=dash),
            marker=dict(symbol=marker, size=7, line=dict(width=0.5, color=COLOR_LAGS)),
            hovertemplate="Lags: %{x}<br>Inference: %{y:.3g} ms<extra>"+label+"</extra>",
        ))

    # --- Spur 2: x = Horizon (bei Lags = l_fix), y = Inference ---
    d_h = dfq[dfq["lags"] == l_fix].copy()
    if not d_h.empty:
        d_h = d_h.sort_values("horizon")
        fig.add_trace(go.Scatter(
            x=d_h["horizon"], y=d_h["avg_inference_time_ms"],
            mode="lines+markers",
            name=f"{label} · Horizon@L{int(l_fix)}",
            legendgroup=f"{label}-hor", showlegend=True,
            line=dict(color=COLOR_HOR, width=2, dash=dash),
            marker=dict(symbol=marker, size=7, line=dict(width=0.5, color=COLOR_HOR)),
            hovertemplate="Horizon: %{x}<br>Inference: %{y:.3g} ms<extra>"+label+"</extra>",
        ))

def plot_time_y_vs_x_lags_horizon(df: pd.DataFrame, algorithm: str) -> go.Figure:
    d = _clean_df(df)
    g = _agg(d[d["algorithm"] == algorithm])
    if g.empty:
        raise ValueError(f"Keine Daten für '{algorithm}'.")

    # verfügbare Fixwerte (snap)
    h_avail = _nearest_available(g["horizon"].unique(), H_FIX)
    l_avail = _nearest_available(g["lags"].unique(), L_FIX)

    # Quant-Overlay-Regel: lstm/cnn1d -> alle; sonst bevorzugt no-quant
    algo_lower = algorithm.lower()
    quants_all = list(g["quant_mode"].unique())
    if algo_lower in ("lstm","cnn1d"):
        quants = [q for q in ("no-quant","quant-fp16","quant-int8") if q in quants_all] or quants_all
    else:
        quants = ["no-quant"] if "no-quant" in quants_all else quants_all

    fig = go.Figure()
    for q in quants:
        dfq = g[g["quant_mode"] == q].copy()
        dfq = dfq[(dfq["horizon"].isin([h_avail])) | (dfq["lags"].isin([l_avail]))]
        if dfq.empty:
            continue
        _add_traces_xy(fig, dfq, q, h_fix=h_avail, l_fix=l_avail)

    note = ""
    if h_avail != H_FIX or l_avail != L_FIX:
        note = f"Hinweis: auf verfügbare Werte gesnappt (H={h_avail}, L={l_avail})."
    _apply_layout(fig, f"{algorithm} · y=Inference, x=Lags@H={H_FIX} & Horizon@L={L_FIX}", note=note)
    return fig

# ---- Ausführen: ein Plot pro Modell ----
algos = sorted(df_out["algorithm"].unique())
for algo in algos:
    fig = plot_time_y_vs_x_lags_horizon(df_out, algorithm=algo)
    fig.show()


In [70]:
# === Plot: X = Horizon, Y1 = MAPE (rot), Y2 = Inference (blau), 3 Levels pro Modell ===
import ast
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FONT_SCI  = dict(family="Arial, Helvetica, sans-serif", size=14)
GRIDCOLOR = "rgba(0,0,0,0.12)"
COLOR_MAPE = "#d62728"   # rot (links)
COLOR_INF  = "#1f77b4"   # blau (rechts)
LEVEL_ORDER = ["simple","medium","high"]
LEVEL_DASH  = {"simple":"solid","medium":"dash","high":"dot"}
LEVEL_MARK  = {"simple":"circle","medium":"square","high":"triangle-up"}

def _parse_series_or_scalar(x):
    if pd.isna(x): return []
    if isinstance(x, (int, float, np.number)): return [float(x)]
    s = str(x).strip()
    try:
        if s.startswith("(") and s.endswith(")"):
            return [float(v) for v in ast.literal_eval(s)]
        return [float(s)]
    except Exception:
        return []

def _clean(df):
    need = {"algorithm","level","horizon","avg_inference_time_ms","metrics_value.mape"}
    miss = need - set(df.columns)
    if miss:
        raise ValueError(f"Fehlende Spalten: {sorted(miss)}")
    d = df.copy()
    d["algorithm"] = d["algorithm"].astype(str)
    d["level"] = d["level"].astype(str).str.strip().str.lower()
    d["horizon"] = pd.to_numeric(d["horizon"], errors="coerce")
    d["avg_inference_time_ms"] = pd.to_numeric(d["avg_inference_time_ms"], errors="coerce")
    return d.dropna(subset=["algorithm","level","horizon","avg_inference_time_ms","metrics_value.mape"])

def _expand_horizons(d):
    rows = []
    for _, r in d.iterrows():
        m_list = _parse_series_or_scalar(r["metrics_value.mape"])
        if not m_list: continue
        for i, m in enumerate(m_list, start=1):
            rows.append({
                "algorithm": r["algorithm"],
                "level": r["level"],
                "horizon_step": i,
                "mape_raw": float(m),
                "inference_ms": float(r["avg_inference_time_ms"]),
            })
    return pd.DataFrame(rows)

def _autoscale_mape_to_percent(series):
    """Gibt (values, label_suffix, was_scaled) zurück.
       Skaliert auf % wenn Median <= 1 (d.h. Werte sind als Anteil gespeichert)."""
    s = pd.Series(series, dtype="float64")
    med = np.nanmedian(s.replace([np.inf, -np.inf], np.nan).dropna())
    if np.isnan(med): return s.values, "", False
    if med <= 1.0:  # vermutlich als Anteil
        return (s * 100.0).values, " [%]", True
    return s.values, " [%]" if med <= 1000 else "", False  # Label: meistens Prozent

def make_levels_plot_for_algorithm_mape(df, algorithm: str, cap_percentile: float = 99.0):
    d = _clean(df)
    d = d[d["algorithm"] == algorithm]
    if d.empty:
        raise ValueError(f"Keine Daten für algorithm='{algorithm}'.")

    exp = _expand_horizons(d)
    if exp.empty:
        raise ValueError("Keine auswertbaren MAPE-Punkte gefunden.")

    # Mittelwert je (level, horizon_step) über Quant/Runs
    g = (exp.groupby(["level","horizon_step"], as_index=False)
             .agg(mape_raw=("mape_raw","mean"), inference_ms=("inference_ms","mean")))

    # Auto-Scaling + optionales Kappen extremer Ausreißer
    mape_vals, mape_unit, scaled = _autoscale_mape_to_percent(g["mape_raw"])
    g = g.assign(mape=mape_vals)
    if cap_percentile is not None:
        cap = np.nanpercentile(g["mape"].replace([np.inf, -np.inf], np.nan).dropna(), cap_percentile)
        if np.isfinite(cap):
            g["mape"] = np.clip(g["mape"], 0, cap)

    levels = [lv for lv in LEVEL_ORDER if lv in set(g["level"])] or sorted(g["level"].unique())

    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for lv in levels:
        sub = g[g["level"] == lv].sort_values("horizon_step")
        if sub.empty: continue
        dash = LEVEL_DASH.get(lv, "solid")
        mark = LEVEL_MARK.get(lv, "circle")

        # MAPE (links)
        fig.add_trace(go.Scatter(
            x=sub["horizon_step"], y=sub["mape"],
            mode="lines+markers",
            name=f"{lv} · MAPE",
            legendgroup=f"{lv}-mape",
            line=dict(color=COLOR_MAPE, width=2, dash=dash),
            marker=dict(symbol=mark, size=7, line=dict(width=0.5, color=COLOR_MAPE)),
            hovertemplate="Horizon: %{x}<br>MAPE: %{y:.3g}"+mape_unit+"<extra>"+lv+"</extra>",
        ), secondary_y=False)

        # Inference (rechts)
        fig.add_trace(go.Scatter(
            x=sub["horizon_step"], y=sub["inference_ms"],
            mode="lines+markers",
            name=f"{lv} · Inference",
            legendgroup=f"{lv}-inf",
            line=dict(color=COLOR_INF, width=2, dash=dash),
            marker=dict(symbol=mark, size=7, line=dict(width=0.5, color=COLOR_INF)),
            hovertemplate="Horizon: %{x}<br>Inference: %{y:.3g} ms<extra>"+lv+"</extra>",
        ), secondary_y=True)

    # Layout
    fig.update_layout(
        title=f"{algorithm} · Levels: simple/medium/high · MAPE & Inference",
        template="simple_white", font=FONT_SCI,
        margin=dict(l=10, r=10, t=60, b=10), height=520,
        legend=dict(orientation="h", y=1.08, x=1, xanchor="right", yanchor="bottom")
    )
    fig.update_xaxes(
        title="Horizon",
        showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
        ticks="outside", showline=True, linewidth=1, linecolor="black"
    )
    fig.update_yaxes(
        title_text=f"MAPE{mape_unit}", secondary_y=False,
        showgrid=True, gridcolor=GRIDCOLOR, zeroline=False,
        ticks="outside", showline=True, linewidth=1, linecolor="black"
    )
    fig.update_yaxes(
        title_text="Inference [ms]", secondary_y=True,
        showgrid=False, zeroline=False,
        ticks="outside", showline=True, linewidth=1, linecolor="black"
    )
    note = "MAPE auto-skaliert auf Prozent" if scaled else "MAPE als übergebene Einheit"
    if cap_percentile is not None:
        note += f" · Ausreißer bis {cap_percentile:.0f}. Perzentil gekappt"
    fig.add_annotation(text=note, xref="paper", yref="paper", x=0, y=1.08,
                       showarrow=False, font=dict(size=12, color="#444"))
    return fig

# ---------- Ausführung: ein Plot pro Modell ----------
# df_levels = <dein DataFrame mit: algorithm, level, horizon, avg_inference_time_ms, metrics_value.mape>
for algo in sorted(df_out["algorithm"].astype(str).unique()):
    fig = make_levels_plot_for_algorithm_mape(df_out, algorithm=algo, cap_percentile=99)
    fig.show()
